Efficiency

Load BESStie

In [1]:
# a lot of ignore warnings were creating noisy outputs, so filter these away.
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

# Load BESSTIE dataset and view overall stats
from datasets import load_dataset
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

dataset = load_dataset("surrey-nlp/BESSTIE-CW-26")

print(dataset)

training_dataframe = dataset['train'].to_pandas()
test_dataframe = dataset['test'].to_pandas()

DatasetDict({
    train: Dataset({
        features: ['text', 'variety', 'source', 'Sentiment', 'Sarcasm'],
        num_rows: 3747
    })
    validation: Dataset({
        features: ['text', 'variety', 'source', 'Sentiment', 'Sarcasm'],
        num_rows: 313
    })
    test: Dataset({
        features: ['text', 'variety', 'source', 'Sentiment', 'Sarcasm'],
        num_rows: 2183
    })
})


Logistic Regression and RoBERTa comparison

In [6]:
import time
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

model_name = "roberta-base"
roberta_tokenizer = AutoTokenizer.from_pretrained(model_name)
roberta_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)
roberta_model.eval()

model_name = "microsoft/deberta-v3-base"

deberta_tokenizer = AutoTokenizer.from_pretrained(model_name)
deberta_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2, dtype="float32").to(device)
deberta_model.eval()

# Timing functions
def transformer_timings(model, tokenizer, texts, device, n_runs=50):
    times = []
    for _ in range(n_runs):
        start = time.time()
        inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=128)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs)
        if device.type == "cuda":
            torch.cuda.synchronize()
        elif device.type == "mps":
            torch.mps.synchronize()
        times.append((time.time() - start) * 1000)
    return np.mean(times), np.std(times)

def svm_timings(model, vectorizer, texts, n_runs=50):
    times = []
    for _ in range(n_runs):
        start = time.time()
        X = vectorizer.transform(texts)
        model.predict(X)
        times.append((time.time() - start) * 1000)
    return np.mean(times), np.std(times)

test_texts = test_dataframe['text'].tolist()
batch_sizes = [1, 16, 32, 64, 128, 256, 512, 1024]

# ---- WARMUP ----
dummy = tokenizer(test_texts[:1], return_tensors="pt", padding=True, truncation=True, max_length=128)
dummy = {k: v.to(device) for k, v in dummy.items()}
with torch.no_grad():
    model(**dummy)
if device.type == "cuda":
    torch.cuda.synchronize()
elif device.type == "mps":
    torch.mps.synchronize()

vectorizer.transform(test_texts[:1])
lr_sentiment.predict(vectorizer.transform(test_texts[:1]))

# ---- TIMING ----
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Apple MPS' if torch.backends.mps.is_available() else 'CPU'}")for size in batch_sizes:
    texts = test_texts[:size]

    svm_mean, svm_std = svm_timings(lr_sentiment, vectorizer, texts)
    rob_mean, rob_std = transformer_timings(roberta_model, roberta_tokenizer, texts, device)
    deb_mean, deb_std = transformer_timings(deberta_model, deberta_tokenizer, texts, device)

    rob_ratio = rob_mean / svm_mean
    deb_ratio = deb_mean / svm_mean

    print(f"Batch {size:>4} | SVM: {svm_mean:>8.2f} ± {svm_std:>6.2f} ms | RoBERTa: {rob_mean:>8.2f} ± {rob_std:>6.2f} ms ({rob_ratio:>6.1f}x) | DeBERTa: {deb_mean:>8.2f} ± {deb_std:>6.2f} ms ({deb_ratio:>6.1f}x)")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier

Batch    1 | SVM:     0.10 ±   0.03 ms | RoBERTa:    12.52 ±  28.80 ms ( 122.6x) | DeBERTa:    23.13 ±  85.79 ms ( 226.4x)
Batch   16 | SVM:     0.33 ±   0.19 ms | RoBERTa:    75.14 ±   5.51 ms ( 227.5x) | DeBERTa:   122.08 ±   6.71 ms ( 369.6x)
Batch   32 | SVM:     0.43 ±   0.06 ms | RoBERTa:   143.26 ±   2.36 ms ( 332.5x) | DeBERTa:   234.19 ±   5.19 ms ( 543.6x)
Batch   64 | SVM:     0.75 ±   0.07 ms | RoBERTa:   282.89 ±   4.62 ms ( 379.4x) | DeBERTa:   462.55 ±   6.71 ms ( 620.4x)
Batch  128 | SVM:     1.48 ±   0.14 ms | RoBERTa:   560.85 ±   3.23 ms ( 378.6x) | DeBERTa:   958.37 ±  54.37 ms ( 646.9x)
Batch  256 | SVM:     2.95 ±   0.26 ms | RoBERTa:  1120.20 ±   7.98 ms ( 379.6x) | DeBERTa:  1964.28 ± 224.13 ms ( 665.7x)
Batch  512 | SVM:     5.58 ±   0.72 ms | RoBERTa:  2261.27 ±  31.15 ms ( 405.5x) | DeBERTa:  4595.23 ± 1068.88 ms ( 824.0x)
Batch 1024 | SVM:    13.19 ±   7.24 ms | RoBERTa:  4695.09 ± 988.86 ms ( 355.9x) | DeBERTa: 57619.45 ± 17915.05 ms (4367.6x)
